In [1]:
from common_setup import *
from pathos.multiprocessing import ProcessingPool as Pool


In [2]:
from InitializeSpeciesPool import *
from LV import *
from VariousMetrics import *

########################################################
########################################################
# Fig 1-1
########################################################
########################################################

##~~~~~~~~~~~~~~ This figure is cartoon ~~~~~~~~~~~~~~##

In [8]:
def estimate_mean_ratio(capacityVar, num_samples=100000, seed=None):
    """
    Estimate E[U/V] where U and V are independent and U = max(N(1, capacityVar), 0.1).

    Parameters:
        capacityVar (float): Variance of the normal distribution.
        num_samples (int): Number of samples to generate for the estimation.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        float: Estimated expected value of U/V.
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Define the mean and standard deviation
    mu = 1
    sigma = np.sqrt(capacityVar)
    
    # Generate samples from N(1, capacityVar)
    U_samples = np.random.normal(mu, sigma, num_samples)
    V_samples = np.random.normal(mu, sigma, num_samples)
    
    # Apply the transformation f_k = max(X, 0.1)
    U_samples = np.maximum(U_samples, 0.1)
    V_samples = np.maximum(V_samples, 0.1)
    
    # Compute the ratios
    ratios = U_samples / V_samples
    
    # Calculate the mean of the ratios
    mean_ratio = np.mean(ratios)
    
    return mean_ratio

estimate_mean_ratio(2)

4.219509508044008

In [10]:
for capacityVar in [0, 0.5, 1.0, 2.0]:
    session_name=f"Simulation_Data/New_k_gaussian_{capacityVar}"
    if not os.path.exists(session_name):
        os.makedirs(session_name)
    print("The new directory is created!")


    ###################
    ######################################
    #########################################################

    N_simul=100
    S=12
    u_list=np.arange(0,1.2,0.1)
    o=0
    t=[0,5000]
    num_C=2
    num_S=12
    N=num_C*num_S
    threshold=1e-3
    f_k = lambda: max(np.random.normal(loc=1, scale=np.sqrt(capacityVar), size=1)[0], 0.1)
    estimated_mean_ratio = estimate_mean_ratio(capacityVar)
    print("estimated_mean_ratio", estimated_mean_ratio)
    u_list = u_list/estimated_mean_ratio
    ###################
    ######################################
    #########################################################
    Intention_to_reset=True
    if Intention_to_reset or not os.path.isfile(session_name+'/Similarity.xlsx'):
        tasks = []
        for i, u in enumerate(u_list):
            for itt in range(N_simul):
                tasks.append((i, itt, u, session_name, t, N, o, threshold, num_C, num_S, f_k))
        
        def simulate_task(task):
            i, itt, u, session_name, t, N, o, threshold, num_C, num_S, f_k = task
            np.random.seed(itt)
            f_interaction = lambda: uniform_distribution(u, o)
            I, g, k = InitializeSpeceiesPool(N, f_interaction, f_g=lambda: np.ones(1),
                                               f_k=f_k, is_diagonal_one=True, save_path=session_name)
            CommunitiesLibrary = InitializeCommunityPool(N, num_C, num_S, I, g, k, save_path=session_name)
            y = np.random.rand(N) * 0.1
            y1 = run_lotka_volterra(y, t, CommunitiesLibrary[0, :], I, g, k)
            y2 = run_lotka_volterra(y, t, CommunitiesLibrary[1, :], I, g, k)
            y1[y1 < threshold] = 0
            y2[y2 < threshold] = 0
            y3 = np.array(y1 + y2)
            survived = y3 > threshold
            y3 = run_lotka_volterra(y3, t, survived, I, g, k)
            y3[y3 < threshold] = 0
            Similarity_to1 = SimilarityTo1(y3/sum(y3), y1/sum(y1), y2/sum(y2), threshold,
                                            lambda x, y, z: SimilarityBC(x, y, z))
            Assimilarity_BC = abs(2*Similarity_to1 - 1)
            a, b, c = metric_VectorDecomposition_onlyPositive(y1, y2, y3)
            return (i, itt, a, b, c)
        
        pool = Pool()
        results = pool.map(simulate_task, tasks)
        pool.close(); pool.join()
        
        Data_1 = np.zeros((len(u_list), N_simul))
        Data_2 = np.zeros((len(u_list), N_simul))
        Data_3 = np.zeros((len(u_list), N_simul))
        for i, itt, a, b, c in results:
            Data_1[i, itt] = a
            Data_2[i, itt] = b
            Data_3[i, itt] = c
        
        df1=pd.DataFrame(Data_1)
        df2=pd.DataFrame(Data_2)
        df3=pd.DataFrame(Data_3)

        with pd.ExcelWriter(session_name+'/Similarity.xlsx') as writer:  
            df1.to_excel(writer,sheet_name='Sheet1')
            df2.to_excel(writer,sheet_name='Sheet2')
            df3.to_excel(writer,sheet_name='Sheet3')

The new directory is created!
estimated_mean_ratio 1.0
Folder 'Simulation_Data/New_k_gaussian_0' already exists.
Folder 'Simulation_Data/New_k_gaussian_0' already exists.
Folder 'Simulation_Data/New_k_gaussian_0' already exists.
Folder 'Simulation_Data/New_k_gaussian_0' already exists.Folder 'Simulation_Data/New_k_gaussian_0' already exists.

Folder 'Simulation_Data/New_k_gaussian_0' already exists.
Folder 'Simulation_Data/New_k_gaussian_0' already exists.Folder 'Simulation_Data/New_k_gaussian_0' already exists.
Folder 'Simulation_Data/New_k_gaussian_0' already exists.

Folder 'Simulation_Data/New_k_gaussian_0' already exists.
Folder 'Simulation_Data/New_k_gaussian_0' already exists.Folder 'Simulation_Data/New_k_gaussian_0' already exists.
Folder 'Simulation_Data/New_k_gaussian_0' already exists.Folder 'Simulation_Data/New_k_gaussian_0' already exists.


Folder 'Simulation_Data/New_k_gaussian_0' already exists.Folder 'Simulation_Data/New_k_gaussian_0' already exists.Folder 'Simulation_

ValueError: Pool not running

In [11]:
results

[(0, 0, 0.7073972352425864, 0.7068158487018235, 0.0007124570624000521),
 (0, 1, 0.7070745959713185, 0.707138133905498, 0.0010841165280890063),
 (0, 2, 0.7072072713297363, 0.7070057284569248, 0.0008805154409192404),
 (0, 3, 0.7070095875477024, 0.7072037355389452, 0.0005652923041083541),
 (0, 4, 0.7070430513790319, 0.7071692488179998, 0.0013330501916496652),
 (0, 5, 0.7070540280598143, 0.7071584042876241, 0.0012620022869731348),
 (0, 6, 0.7071801063922172, 0.7070329181512075, 0.0008658947279987822),
 (0, 7, 0.7069603223225218, 0.7072521410658111, 0.0012294793459042745),
 (0, 8, 0.7068692578038783, 0.7073438401290305, 0.0007377013696167797),
 (0, 9, 0.7071613572415463, 0.7070513070228174, 0.0011243049496864493),
 (0, 10, 0.7068168849643603, 0.707395677952602, 0.0011162182835880597),
 (0, 11, 0.70713449740529, 0.7070780226541925, 0.001213449135737747),
 (0, 12, 0.7068555999187928, 0.7073560605646678, 0.0016013887339905019),
 (0, 13, 0.707566960820471, 0.7066447370678381, 0.0014871212485049